In [9]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()  # membaca isi file .env

API_KEY = os.getenv("Movie_API_KEY")

if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


In [10]:
# Muat file .env
load_dotenv()
API_KEY = os.getenv("Movie_API_KEY")
alamat_api = "https://api.themoviedb.org/3/discover/movie"
parameter = {"api_key": API_KEY, "language": "en-US"}

def ambil_mapping_otomatis(api_key):
    url = "https://api.themoviedb.org/3/genre/movie/list"
    params = {"api_key": api_key, "language": "en-US"}
    response = requests.get(url, params=params)
    
    mapping = {}
    if response.status_code == 200:
        for item in response.json().get("genres", []):
            mapping[item["name"].lower()] = item["id"]
            
    return mapping

response = requests.get(alamat_api, params=parameter)
print(f"Status Code: {response.status_code}")   

hasil = response.json()
# Cek status dan isi teks mentahnya
mapping_genre_otomatis = ambil_mapping_otomatis(API_KEY)
print("Mapping genre:", mapping_genre_otomatis)


Status Code: 200
Mapping genre: {'action': 28, 'adventure': 12, 'animation': 16, 'comedy': 35, 'crime': 80, 'documentary': 99, 'drama': 18, 'family': 10751, 'fantasy': 14, 'history': 36, 'horror': 27, 'music': 10402, 'mystery': 9648, 'romance': 10749, 'science fiction': 878, 'tv movie': 10770, 'thriller': 53, 'war': 10752, 'western': 37}


In [11]:
film_pertama = hasil["results"][10]

print("Judul       :", film_pertama["title"])
print("Release Date:", film_pertama["release_date"])
print("Rating      :", film_pertama["vote_average"])
print("Popularity  :", film_pertama["popularity"])
print("Overview    :", film_pertama["overview"])

Judul       : Moana
Release Date: 2026-07-08
Rating      : 7.425
Popularity  : 183.5472
Overview    : Teenage Moana answers the Ocean's call and, for the first time, voyages beyond the reef of her island of Motunui with infamous demigod Maui on an unforgettable journey to restore prosperity to her people.


In [ ]:
class film:

    def __init__(self, api_key):
        # Kartu identitas disimpan di sini, supaya semua method di bawah bisa memakainya
        self.api_key = api_key
        self.alamat_api= "https://api.themoviedb.org/3/discover/movie"

    def ambil_film(self, genre, halaman):
        parameter = {
             "language"    : "en-US", 
             "page"        : halaman,
             "api_key"     : self.api_key,
             "with_genres" : str(genre_id),
             "sort_by"     : "popularity.desc" 
         }

        try:
            # Rencana utama: menghubungi NewsAPI
            # timeout=20 artinya kita hanya sabar menunggu 20 detik
            response = requests.get(self.alamat_api, params=parameter, timeout=20)
        except Exception:
            # Rencana cadangan: kalau koneksi bermasalah, kita coba sekali lagi
            print("Koneksi bermasalah, mencoba lagi...")
            time.sleep(3)
            response = requests.get(self.alamat_api, params=parameter, timeout=20)

        if response.status_code != 200:
            print(f"Gagal mengambil data. Status: {response.status_code}")
            return pd.DataFrame()

        daftar_film = response.json()["results"]

        data = []
        for film in daftar_film:
            data.append({
                "Judul"        : film["title"],
                "genre"        : film["genre_ids"],
                "bahasa"       : film["original_language"],
                "release"      : film["release_date"],
                "rating"       : film["vote_average"],
                "penonton"      : film["popularity"]
            })

        return pd.DataFrame(data)

mapping_genre_otomatis = ambil_mapping_otomatis(API_KEY)

# 3. Inisialisasi menggunakan class baru
print("Class film siap di gunakan!")


Class film siap di gunakan!


In [6]:
hasil = response.json()

print(f"Jumlah film ditemukan: {hasil['total_results']}")
print(f"Jumlah film perhalaman yang di kirim: {len(hasil['results'])}")

# Masukkan hasil film ke DataFrame
df = pd.DataFrame(hasil["results"])

print(df[["id", "title", "vote_average", "release_date"]].head())

Jumlah film ditemukan: 20001
Jumlah film perhalaman yang di kirim: 20
        id                      title  vote_average release_date
0   969681  Spider-Man: Brand New Day         7.862   2026-07-29
1  1423191              Resident Evil         7.328   2026-09-16
2  1101383      The End of Oak Street         7.009   2026-08-12
3  1204680            Coyote vs. Acme         7.536   2026-08-20
4  1368337                The Odyssey         8.017   2026-07-15


In [7]:
klien = film(API_KEY)

# 4. Daftar kata kunci string bebas
daftar_kata_kunci = ["horror", "action","adventure"] 
semua_tabel = []

for kata_kunci in daftar_kata_kunci:
    key_bersih = kata_kunci.lower().strip()
    genre_id = mapping_genre_otomatis.get(key_bersih)
    
    if not genre_id:
        print(f"\nGenre '{kata_kunci}' tidak ditemukan di TMDB.")
        continue
        
    print(f"\n--- Mengambil film POPULER untuk: '{kata_kunci}' (ID: {genre_id}) ---")
    semua_data_keyword = []
    halaman = 1  
    
    while halaman <=10:
        tabel = klien.ambil_film(genre_id, halaman)
        
        if tabel.empty:
            print(f"Halaman {halaman} kosong atau habis.")
            break
            
        print(f"Halaman {halaman}: Berhasil mengambil {len(tabel)} film")
        semua_data_keyword.append(tabel)
        
        halaman += 1  
        time.sleep(0.2)
        
    if semua_data_keyword:
        tabel_gabung = pd.concat(semua_data_keyword, ignore_index=True)
        semua_tabel.append(tabel_gabung)
        print(f"Total terkumpul untuk '{kata_kunci}': {len(tabel_gabung)} film")
        
# 5. Gabungkan dan ringkas kolomnya agar rapi

if semua_tabel:
    df_film = pd.concat(semua_tabel, ignore_index=True)
    
    # buat dictionary untuk id ke ubah nama genre
    id_ubah_nama = {id: nama for nama,
                    id in mapping_genre_otomatis.items()
                    }

    df_film['genre'] = df_film['genre'].apply(
    lambda daftar_id: ", ".join([id_ubah_nama.get(i, str(i)) for i in daftar_id])
)
    print(f"\n========================================")
    print(f"Total keseluruhan film terkumpul: {len(df_film)}")
    print(f"========================================")
    display(df_film.head(10))
else:
    print("\nTidak ada data sama sekali yang berhasil dikumpulkan.")


--- Mengambil film POPULER untuk: 'horror' (ID: 27) ---


Halaman 1: Berhasil mengambil 20 film
Halaman 2: Berhasil mengambil 20 film
Halaman 3: Berhasil mengambil 20 film
Halaman 4: Berhasil mengambil 20 film
Halaman 5: Berhasil mengambil 20 film
Halaman 6: Berhasil mengambil 20 film
Halaman 7: Berhasil mengambil 20 film
Halaman 8: Berhasil mengambil 20 film
Halaman 9: Berhasil mengambil 20 film
Halaman 10: Berhasil mengambil 20 film
Total terkumpul untuk 'horror': 200 film

--- Mengambil film POPULER untuk: 'action' (ID: 28) ---
Halaman 1: Berhasil mengambil 20 film
Halaman 2: Berhasil mengambil 20 film
Halaman 3: Berhasil mengambil 20 film
Halaman 4: Berhasil mengambil 20 film
Halaman 5: Berhasil mengambil 20 film
Halaman 6: Berhasil mengambil 20 film
Halaman 7: Berhasil mengambil 20 film
Halaman 8: Berhasil mengambil 20 film
Halaman 9: Berhasil mengambil 20 film
Halaman 10: Berhasil mengambil 20 film
Total terkumpul untuk 'action': 200 film

--- Mengambil film POPULER untuk: 'adventure' (ID: 12) ---
Halaman 1: Berhasil mengambil 20 film
H

,Judul,genre,bahasa,release,rating,penoton
0,Resident Evil,"horror, science fiction, adventure",en,2026-09-16,7.341,607.7799
1,Colony,"action, horror, science fiction",ko,2026-05-21,8.100,318.7187
2,Obsession,"horror, thriller",en,2026-05-13,8.194,142.2624
3,Pinocchio: Unstrung,"horror, fantasy, mystery",en,2026-07-22,6.565,138.0530
4,Project Sacrifice,"horror, thriller",id,2026-05-13,3.917,114.4236
5,Teenage Sex and Death at Camp Miasma,"comedy, horror, romance",en,2026-08-06,6.456,101.3883
6,Dark Nuns,"horror, mystery, thriller, drama",ko,2025-01-24,6.186,99.5727
7,Ghost in the Cell,"horror, comedy, thriller",id,2026-04-16,7.200,95.9648
8,Deep Water,"horror, thriller, adventure",en,2026-04-30,7.357,91.2568
9,Backrooms,"horror, mystery, science fiction",en,2026-05-27,7.033,90.2978


In [8]:
print("1. Jumlah sel kosong per kolom:")
print(df_film.isnull().sum())
print()

print("2. Jumlah baris yang kembar (berdasarkan URL):")
print(df_film.duplicated(subset=["Judul", "release"]).sum())
print()

print("3. Tipe data setiap kolom:")
print(df_film.dtypes)

1. Jumlah sel kosong per kolom:
Judul      0
genre      0
bahasa     0
release    0
rating     0
penoton    0
dtype: int64

2. Jumlah baris yang kembar (berdasarkan URL):
121

3. Tipe data setiap kolom:
Judul          str
genre          str
bahasa         str
release        str
rating     float64
penoton    float64
dtype: object


In [9]:
baris_duplikat = df_film[df_film.duplicated(subset=["Judul", "release"], keep=False)]
baris_duplikat = baris_duplikat.sort_values(by="Judul")
display(baris_duplikat)

,Judul,genre,bahasa,release,rating,penoton
178,A Quiet Place: Day One,"horror, science fiction, thriller",en,2024-06-26,6.631,17.2698
180,A Quiet Place: Day One,"horror, science fiction, thriller",en,2024-06-26,6.631,17.2698
583,Ant-Man,"science fiction, adventure, action",en,2015-07-14,7.100,27.8134
388,Ant-Man,"science fiction, adventure, action",en,2015-07-14,7.067,27.8134
379,Ant-Man and the Wasp: Quantumania,"action, adventure, science fiction",en,2023-02-15,6.200,28.3314
...,...,...,...,...,...,...
35,World War Z,"action, horror, science fiction",en,2013-06-19,6.851,32.5965
503,X-Men: Days of Future Past,"action, adventure, science fiction",en,2014-05-15,7.537,36.5255
298,X-Men: Days of Future Past,"action, adventure, science fiction",en,2014-05-15,7.537,36.5255
580,Zack Snyder's Justice League,"action, adventure, fantasy",en,2021-03-18,8.074,27.9553


In [10]:
def bersihkan_deskripsi(teks):
    if pd.isna(teks):
        return "Tidak ada deskripsi"
    return teks


df_film["Judul"] = df_film["Judul"].apply(bersihkan_deskripsi)

# Baris tanpa judul kita buang, karena berita seperti itu tidak berguna
jumlah_sebelum = len(df_film)
df_film = df_film.dropna(subset=["Judul"])
print(f"Baris tanpa judul yang dibuang: {jumlah_sebelum - len(df_film)}")

print()
print("Sel kosong setelah ditangani:")
print(df_film.isnull().sum())

Baris tanpa judul yang dibuang: 0

Sel kosong setelah ditangani:
Judul      0
genre      0
bahasa     0
release    0
rating     0
penoton    0
dtype: int64


In [11]:
jumlah_sebelum = len(df_film)

df_bersih = df_film.drop_duplicates(subset="Judul")

print(f"Jumlah baris sebelum : {jumlah_sebelum}")
print(f"Jumlah baris sesudah : {len(df_bersih)}")
print(f"Baris kembar dibuang : {jumlah_sebelum - len(df_bersih)}")

Jumlah baris sebelum : 600
Jumlah baris sesudah : 474
Baris kembar dibuang : 126


In [12]:
def ubah_ke_tanggal(teks):
    return pd.to_datetime(teks)


print("Tipe data sebelum:", df_bersih["release"].dtype)

df_bersih["release"] = df_bersih["release"].apply(ubah_ke_tanggal)

print("Tipe data sesudah:", df_bersih["release"].dtype)
df_bersih.head()

Tipe data sebelum: str
Tipe data sesudah: datetime64[us]


,Judul,genre,bahasa,release,rating,penoton
0,Resident Evil,"horror, science fiction, adventure",en,2026-09-16,7.341,607.7799
1,Colony,"action, horror, science fiction",ko,2026-05-21,8.100,318.7187
2,Obsession,"horror, thriller",en,2026-05-13,8.194,142.2624
3,Pinocchio: Unstrung,"horror, fantasy, mystery",en,2026-07-22,6.565,138.0530
4,Project Sacrifice,"horror, thriller",id,2026-05-13,3.917,114.4236


In [13]:
print(f"Jumlah baris           : {len(df_bersih)}")
print(f"Sudah lebih dari 100?  : {len(df_bersih) >= 100}")
print(f"Judul masih ada kosong : {df_bersih['Judul'].isnull().sum()}")
print(f"URL masih kembar       : {df_bersih['Judul'].duplicated().sum()}")
print(f"Tipe kolom Tanggal     : {df_bersih['release'].dtype}")

Jumlah baris           : 474
Sudah lebih dari 100?  : True
Judul masih ada kosong : 0
URL masih kembar       : 0
Tipe kolom Tanggal     : datetime64[us]


In [14]:
df_bersih.to_csv("dataset_film.csv", index=False)
print("Data berhasil disimpan ke file: dataset_berita.csv")

df_cek = pd.read_csv("dataset_film.csv")
print(f"File terbaca kembali: {len(df_cek)} baris, {len(df_cek.columns)} kolom")
df_cek.head()

Data berhasil disimpan ke file: dataset_berita.csv
File terbaca kembali: 474 baris, 6 kolom


,Judul,genre,bahasa,release,rating,penoton
0,Resident Evil,"horror, science fiction, adventure",en,2026-09-16,7.341,607.7799
1,Colony,"action, horror, science fiction",ko,2026-05-21,8.100,318.7187
2,Obsession,"horror, thriller",en,2026-05-13,8.194,142.2624
3,Pinocchio: Unstrung,"horror, fantasy, mystery",en,2026-07-22,6.565,138.0530
4,Project Sacrifice,"horror, thriller",id,2026-05-13,3.917,114.4236


In [15]:
print("=" * 50)
print("ANGKA UNTUK SLIDE")
print("=" * 50)
print(f"Sumber data      : themoviedb.org")
print(f"Kata kunci dipakai: {', '.join(daftar_kata_kunci)}")
print()
print(f"Baris sebelum dibersihkan : {len(df_film) + (jumlah_sebelum - len(df_bersih))}")
print(f"Baris dataset akhir       : {len(df_bersih)}")
print()
print("Class yang dibuat:")
print("  1. film - mengambil data berita dari themoviedbAPI")
print()
print("Function yang dibuat:")
print("  1. bersihkan_deskripsi - mengisi deskripsi kosong dengan penanda")
print("  2. ubah_ke_tanggal     - mengubah tulisan tanggal jadi tipe tanggal")
print()
print("Temuan dari pembersihan data:")
print(f"  Baris kembar dibuang     : {jumlah_sebelum - len(df_bersih)}")
print(f"  Deskripsi kosong diberi penanda: ada")
print("=" * 50)

ANGKA UNTUK SLIDE
Sumber data      : themoviedb.org
Kata kunci dipakai: horror, action, adventure

Baris sebelum dibersihkan : 726
Baris dataset akhir       : 474

Class yang dibuat:
  1. film - mengambil data berita dari themoviedbAPI

Function yang dibuat:
  1. bersihkan_deskripsi - mengisi deskripsi kosong dengan penanda
  2. ubah_ke_tanggal     - mengubah tulisan tanggal jadi tipe tanggal

Temuan dari pembersihan data:
  Baris kembar dibuang     : 126
  Deskripsi kosong diberi penanda: ada
